# NEXTQUESTION — cost-aware information gathering for fictional clinical cases

**Veritas AI International High School Hackathon 2026 · AI + Health**

**Team:** Abdulla Alhajaj · Zayed Alketbi · Mohammed Alali

This is an educational demonstration of an AI that selects informative **synthetic diagnostic test findings** across several clinical categories. It includes a **separate pilot based on genuine, de-identified, publicly published MIMIC-IV hospital records** and a public symptom benchmark. Real-record results must not be confused with simulated performance. No use with actual patients is permitted.

**This project is a prototype and is not medical advice.** Never enter real patient information or use this demo to decide what medical tests to order or omit.

Use **Runtime → Run all**. After the tests and research reports finish, open the temporary `gradio.live` link generated in the last cell. Keep this Colab session running.



In [ ]:
import os, sys, subprocess
repo = "/content/NEXTQUESTION-Veritas-2026"
url = "https://github.com/ahblxz/NEXTQUESTION-Veritas-2026.git"
if os.path.isdir(repo + "/.git"):
    subprocess.run(["git", "-C", repo, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", repo, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, repo], check=True)
os.chdir(repo)
if repo not in sys.path:
    sys.path.insert(0, repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("NEXTQUESTION multi-condition dependencies installed.")



In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_hospital.py", "tests/test_audit.py"], check=True)
subprocess.run([sys.executable, "hospital_train.py", "--out",
                "artifacts/hospital", "--n", "4800"], check=True)
subprocess.run([sys.executable, "-m", "hospital.audit"], check=True)
subprocess.run([sys.executable, "-m", "hospital.public_benchmark"], check=True)
subprocess.run([sys.executable, "-m", "hospital.mimic_real"], check=True)
print("Synthetic, independent public symptom and real public MIMIC EHR studies reproduced. The real-record model is a separate retrospective pilot.")


In [ ]:
from IPython.display import display, Markdown
from pathlib import Path
audit = Path("artifacts/hospital/audit_report.md").read_text(encoding="utf-8")
public = Path("artifacts/hospital/public/README.md").read_text(encoding="utf-8")
display(Markdown("\n".join(audit.splitlines()[:42])))
display(Markdown(public))
import json
mimic = json.loads(Path("artifacts/mimic_real/real_metrics.json").read_text(encoding="utf-8"))
display(Markdown(
    "### Real public de-identified EHR pilot (different experiment)\n"
    f"**{mimic['split_admissions']['test']}** held-out admissions from "
    f"**{mimic['split_patients']['test']}** unseen patients. "
    f"Adaptive accuracy: **{mimic['adaptive']['top1_accuracy']:.1%}**, "
    f"fixed-order accuracy: **{mimic['fixed']['top1_accuracy']:.1%}**. "
    "The adaptive policy underperformed and the data cannot establish "
    "clinically unnecessary testing or cost savings."
))


In [ ]:
import importlib, sys
if "app" in sys.modules:
    try:
        sys.modules["app"].demo.close()
    except Exception:
        pass
    hospital_app = importlib.reload(sys.modules["app"])
else:
    import app as hospital_app
print("Open the temporary public gradio.live link below. This is an educational synthetic research simulator.")
hospital_app.demo.launch(share=True, show_error=True, theme=hospital_app.THEME, css=hospital_app.CSS)

